In [47]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('week1\insurance.csv')
print(df.head())
print(df.shape)

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520
(1338, 7)


In [48]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [49]:
print(df.isnull().sum())

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


In [50]:
#BMI categories 

df['bmi_category'] = pd.cut(df['bmi'],
                            bins = [0, 18.5, 24.9, 29.9, 100],
                            labels = ['underweight', 'normal','overweight', 'obese'])

print(df['bmi_category'].value_counts(normalize=True). round(1) * 100)


bmi_category
obese          50.0
overweight     30.0
normal         20.0
underweight     0.0
Name: proportion, dtype: float64


In [51]:
#risk tier
def assign_risk_tier(row):
    if row['smoker'] == 'yes' and row['bmi_category'] == 'obese':
        return 'Very High'
    elif row['smoker'] == 'yes':
        return 'High'
    elif row['bmi_category'] == 'obese':
        return 'Medium'
    else:
        return 'Low' 

df['risk_tier'] = df.apply(assign_risk_tier, axis=1)
print(df['risk_tier'].value_counts())

risk_tier
Medium       569
Low          495
Very High    147
High         127
Name: count, dtype: int64


In [52]:
#Average charges by risk tier
avg_charge_risktier = df.groupby('risk_tier')['charges'].mean().round(2).sort_values(ascending=False)
print(avg_charge_risktier)

risk_tier
Very High    41355.87
High         21279.14
Medium        8809.55
Low           8002.89
Name: charges, dtype: float64


In [53]:
#Average charges by region and risk tier 
charge_region_risktier = df.groupby(['region', 'risk_tier'])['charges'].mean().round(2).unstack()
print(charge_region_risktier)

risk_tier      High      Low    Medium  Very High
region                                           
northeast  20775.26  8276.99  10280.10   40648.08
northwest  22152.99  8076.00   9086.10   42425.28
southeast  21534.42  7901.64   8089.26   42064.29
southwest  20404.78  7684.83   8331.93   40065.59


In [54]:
#Age category
df['age_category'] = pd.cut(df['age'], 
                        bins = [16, 25, 40, 65],
                        labels = ['Young (16-25)', 'Adult (26-40)', 'Middle Age (41-65)'])
print((df['age_category'].value_counts(normalize=True) * 100).round(1))

age_category
Middle Age (41-65)    47.6
Adult (26-40)         29.5
Young (16-25)         22.9
Name: proportion, dtype: float64


In [55]:
print(df['age'].describe())
print(df[df['age'] <= 25]['age'].count())

count    1338.000000
mean       39.207025
std        14.049960
min        18.000000
25%        27.000000
50%        39.000000
75%        51.000000
max        64.000000
Name: age, dtype: float64
306


In [56]:
#Average charges by age and risk tier 
charge_age_risktier = df.groupby(['age_category', 'risk_tier'],observed=True)['charges'].mean().round(2).unstack()
print(charge_age_risktier)

risk_tier               High       Low    Medium  Very High
age_category                                               
Young (16-25)       17606.09   3939.02   4070.46   36252.91
Adult (26-40)       18966.63   6201.19   6267.85   39734.66
Middle Age (41-65)  25023.80  11735.91  11941.32   44929.81


In [59]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder 
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np 

In [63]:
df.head()

,age,sex,bmi,children,smoker,region,charges,bmi_category,risk_tier,age_category
0,19,female,27.900,0,yes,southwest,16884.92400,overweight,High,Young (16-25)
1,18,male,33.770,1,no,southeast,1725.55230,obese,Medium,Young (16-25)
2,28,male,33.000,3,no,southeast,4449.46200,obese,Medium,Adult (26-40)
3,33,male,22.705,0,no,northwest,21984.47061,normal,Low,Adult (26-40)
4,32,male,28.880,0,no,northwest,3866.85520,overweight,Low,Adult (26-40)


In [67]:
df_model = df.copy()

le = LabelEncoder()
df_model['sex_encoded'] = le.fit_transform(df_model['sex'])
df_model['smoker_encoded'] = le.fit_transform(df_model['smoker'])
df_model['region_encoded'] = le.fit_transform(df_model['region'])
df_model['bmi_category_encoded'] = le.fit_transform(df_model['bmi_category'])
df_model['risk_tier_encoded'] = le.fit_transform(df_model['risk_tier'])

In [70]:
# Defining X and y
X = df_model[['age', 'bmi', 'children','smoker_encoded', 
              'region_encoded','bmi_category_encoded', 'risk_tier_encoded']]

y = df_model['charges']

print(X.shape)
print(y.shape)

(1338, 7)
(1338,)


In [72]:
# Split into training and test sets

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state =42)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")


Training set: 1070 rows
Test set: 268 rows


In [75]:
#Train model 
model = LinearRegression()
model.fit(X_train, y_train)

#Evaluate 
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R² Score: {round(r2, 3)}")
print(f"Mean Absolute Error: ${round(mae, 2)}")

R² Score: 0.84
Mean Absolute Error: $3207.67


In [76]:
# Feature importance
feature_names = X.columns 
coefficients = model.coef_

importance = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients}).sort_values('Coefficient', ascending=False)

print(importance)

                Feature   Coefficient
3        smoker_encoded  23235.688800
6     risk_tier_encoded   6225.091131
5  bmi_category_encoded    921.487370
2              children    415.748472
0                   age    257.593614
1                   bmi   -255.484680
4        region_encoded   -345.371213
